In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/durgashyam9849@gmail.com/consolidated_pipeline/setup/utilities

In [0]:
dbutils.widgets.text('catalog','fmcg','Catalog')
dbutils.widgets.text('data_source', 'orders', 'Data Source')

catalog = dbutils.widgets.get('catalog')
data_source = dbutils.widgets.get('data_source')

base_path = f's3://sportsbar-dem/{data_source}'
landing_path = f'{base_path}/landing/'
processed_path = f'{base_path}/processed/'

bronze_table = f'{catalog}.{bronze_schema}.{data_source}'
silver_table = f'{catalog}.{silver_schema}.{data_source}'
gold_table = f'{catalog}.{gold_schema}.sb_fact_{data_source}'

print(base_path)
print(landing_path)
print(processed_path)

In [0]:
df = spark.read.options(header=True, inferSchema=True).csv(f'{landing_path}/*.csv').withColumn('read_timestamp', F.current_timestamp()).select('*', '_metadata.file_name', '_metadata.file_size')

In [0]:
df.count()

In [0]:
display(df.limit(20))

In [0]:
df.write\
    .format('delta')\
    .mode('append')\
    .option('delta.enableChangeDataFeed', 'true')\
    .saveAsTable(bronze_table)

In [0]:
files = dbutils.fs.ls(landing_path)
files

for file_info in files:
  dbutils.fs.mv(file_info.path, f'{processed_path}/{file_info.name}', True)

# Silver

In [0]:
df_orders = spark.sql(f'SELECT * FROM {bronze_table}')
display(df_orders.limit(5))

# without function

In [0]:
# df_orders = df_orders.filter(F.col('order_qty').isNotNull())

# df_orders = df_orders.withColumn('customer_id',
#     F.when(F.col('customer_id').rlike('^[0-9]+$'), F.col('customer_id')).otherwise('999999').cast('string'))

# df_orders = df_orders.withColumn('order_placement_date', F.regexp_replace(F.col('order_placement_date'), r'^[A-Za-z]+,\s*',''))

# df_orders = df_orders.withColumn('order_placement_date', 
#             F.coalesce(F.try_to_date(F.col('order_placement_date'), 'yyyy/MM/dd'),
#             F.try_to_date(F.col('order_placement_date'), 'dd/MM/yyyy'),
#             F.try_to_date(F.col('order_placement_date'), 'MMMM dd, yyyy'),
#             F.try_to_date(F.col('order_placement_date'), 'dd-MM-yyyy')))

# df_orders = df_orders.dropDuplicates(['order_id', 'customer_id', 'product_id', 'order_qty','order_placement_date'])

# df_orders = df_orders.withColumn('product_id', F.col('product_id').cast('string'))


#With Function

In [0]:
def clean_orders(df_orders):
    return (
        df_orders
            .filter(F.col('order_qty').isNotNull())
            .withColumns({
                'customer_id': F.when(F.col('customer_id').rlike('^[0-9]+$'), F.col('customer_id')).otherwise('999999').cast('string'),
            'order_placement_date': F.coalesce(
                    F.try_to_date(F.regexp_replace(F.col('order_placement_date'), r'^[A-Za-z]+,\s*',''), 'yyyy/MM/dd'),
                    F.try_to_date(F.regexp_replace(F.col('order_placement_date'), r'^[A-Za-z]+,\s*',''), 'dd/MM/yyyy'),
                    F.try_to_date(F.regexp_replace(F.col('order_placement_date'), r'^[A-Za-z]+,\s*',''), 'MMMM dd, yyyy'),
                    F.try_to_date(F.regexp_replace(F.col('order_placement_date'), r'^[A-Za-z]+,\s*',''), 'dd-MM-yyyy'),
                )})
            .dropDuplicates(['order_id', 'customer_id', 'product_id', 'order_qty', 'order_placement_date'])
            .withColumn('product_id', F.col('product_id').cast('string'))
    )

df_orders = clean_orders(df_orders)

In [0]:
df_min_max_date = df_orders.agg(
    F.min('order_placement_date').alias('min_date'),
    F.max('order_placement_date').alias('max_date')
).show()

In [0]:
display(df_orders.limit(2000))

In [0]:
df_products = spark.table(f'fmcg.silver.products')
display(df_products.limit(5))

In [0]:
df_joined = df_orders.join(df_products, on='product_id', how='inner').select(df_orders['*'], df_products['product_code'])
display(df_joined.limit(10))

In [0]:
if not (spark.catalog.tableExists(silver_table)):
    df_joined.write.format('delta').option('delta.enableChangeDataFeed', 'true').option('mergeSchema', 'true').mode('overwrite').saveAsTable(silver_table)
else:
    silver_delta = DeltaTable.forName(spark,silver_table)
    silver_delta.alias('silver').merge(
        df_joined.alias('bronze'),
        'silver.order_placement_date = bronze.order_placement_date and silver.order_id = bronze.order_id and silver.product_code = bronze.product_code and silver.customer_id = bronze.customer_id').whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

#Gold

In [0]:
df_gold = spark.sql(f'select order_id, order_placement_date as date, customer_id as customer_code, product_code, product_id, order_qty as sold_quantity from {silver_table}')
display(df_gold.limit(5))

In [0]:
if not (spark.catalog.tableExists(gold_table)):
    print('creating new table')
    df_gold.write.format('delta').option('delta.enableChangeDataFeed', 'true').option('mergeSchema', 'true').mode('overwrite').saveAsTable(gold_table)
else:
    gold_delta = DeltaTable.forName(spark,silver_table)
    gold_delta.alias('source').merge(
        df_gold.alias('gold'),
        'source.date = gold.ate and source.order_id = gold.order_id and source.product_code = gold.product_code and source.customer_code = gold.customer_code').whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

#Merge with parent company(table)

In [0]:
df_child = spark.sql(f'select date, product_code, customer_code, sold_quantity from {gold_table}')
display(df_child.limit(10))

In [0]:
df_child.count()

In [0]:
df_monthly = (df_child
              .withColumn('month_start', F.trunc('date','MM'))
              .groupBy('month_start', 'product_code', 'customer_code')
              .agg(F.sum('sold_quantity').alias('sold_quantity'))
              .withColumnRenamed("month_start", "date")
              )

In [0]:
df_monthly.show(500, truncate=False)

In [0]:
df_monthly.count()

In [0]:
gold_parent_delta = DeltaTable.forName(spark, f"{catalog}.{gold_schema}.fact_orders")
gold_parent_delta.alias("parent_gold").merge(df_monthly.alias("child_gold"), "parent_gold.date = child_gold.date AND parent_gold.product_code = child_gold.product_code AND parent_gold.customer_code = child_gold.customer_code").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()